# Visualização exploratória — renda por bairro

Este notebook serve para **pré-visualizar** o mapa de renda antes de decidir o estilo final das figuras do piloto.

Ele lê o GeoPackage gerado por `scripts/downloads.py`, mostra o gradiente de renda e escreve o nome de **todos os bairros**. A visualização **não é salva automaticamente** em PNG.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt

# Localiza a raiz do projeto mesmo executando a partir de notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualization import preview_income_map_with_labels

DATA_DIR = PROJECT_ROOT / "data" / "censo_2022"
GPKG_PATH = DATA_DIR / "bairros_setores_renda.gpkg"
BOUNDARY_PATH = DATA_DIR / "limite_poa.gpkg"

print("Raiz do projeto:", PROJECT_ROOT)
print("GeoPackage:", GPKG_PATH)

In [ ]:
if not GPKG_PATH.exists():
    raise FileNotFoundError(
        "O GeoPackage ainda não existe. Execute primeiro: "
        "python scripts/downloads.py"
    )

bairros = gpd.read_file(
    GPKG_PATH,
    layer="bairros_renda",
)

limite = None
if BOUNDARY_PATH.exists():
    limite = gpd.read_file(
        BOUNDARY_PATH,
        layer="limite_poa",
    )

print(f"Bairros carregados: {len(bairros)}")
print("CRS:", bairros.crs)
print("Colunas:")
print(list(bairros.columns))

In [ ]:
# Verificação rápida dos dados que serão representados
bairros[["NM_BAIRRO", "RENDA_MED_BAIRRO"]].sort_values(
    "RENDA_MED_BAIRRO"
).reset_index(drop=True)

In [ ]:
# Parâmetros exploratórios: altere livremente para comparar alternativas.
CMAP = "YlOrRd"
FIGSIZE = (16, 16)
LABEL_FONTSIZE = 5.5

fig, ax = preview_income_map_with_labels(
    neighborhoods=bairros,
    value_column="RENDA_MED_BAIRRO",
    name_column="NM_BAIRRO",
    title="Distribuição da renda média por bairro — Porto Alegre",
    cmap=CMAP,
    boundary=limite,
    figsize=FIGSIZE,
    label_fontsize=LABEL_FONTSIZE,
)

# Em Jupyter, deixar a figura como última expressão exibe o mapa inline.
# Nenhum arquivo é salvo por esta célula.
fig

## O que observar antes de definir o mapa final

- legibilidade dos nomes em bairros pequenos;
- comportamento do gradiente nas rendas extremas;
- necessidade ou não de quebrar os valores em classes;
- tamanho ideal dos rótulos;
- necessidade de reposicionar manualmente alguns nomes.

Depois dessa inspeção, os parâmetros escolhidos podem ser incorporados ao mapa exportado pelo pipeline.